In [ ]:
# =============================================================================
#  SHROOM-visions 2026 — text-only span tagger
#  XLM-R token classification -> per-character hallucination probability
#
#  Kaggle: T4, ~20 min end to end. Expects the data already extracted at
#  /kaggle/working/distrib/ (as in the earlier notebook).
#
#  Produces: dev scores + predictions_{lang}.jsonl.
# =============================================================================

In [ ]:
ls /kaggle/working/distrib

In [ ]:
# ── Cell 0: fetch data if not already present ────────────────────────────────
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
if not os.path.exists("/kaggle/working/distrib"):
    !wget -q https://a3s.fi/mickusti-2007780-pub/shroom-visions-data.zip -O /tmp/data.zip
    !unzip -oq /tmp/data.zip -d /kaggle/working/
print(sorted(os.listdir("/kaggle/working/distrib")))

In [ ]:
# ── Cell 1: setup ────────────────────────────────────────────────────────────
import json, os, random, pathlib
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from scipy.stats import spearmanr

IMAGE_DIR = "/kaggle/working/shroom-vis-images"
DISTRIB    = "/kaggle/working/distrib"
OUT_DIR    = "/kaggle/working"
MODEL_ID   = "xlm-roberta-large"
LANGS      = ["en", "fr", "it", "zh"]
CATEGORIES = ["invention", "mischaracterization", "OCR", "miscounting", "other"]
MAX_LEN    = 256
BATCH      = 8
EPOCHS     = 5
LR         = 1e-5
SEED       = 13

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
# ── Cell 2: official scoring functions ───────────────────────────────────────
# Copied verbatim from the organizers' scorer.py so dev numbers match the
# leaderboard exactly. Do not edit.

def score_cor(ref_dict, pred_dict, label_filtered_=None):
    assert ref_dict['id'] == pred_dict['id']
    ref_vec = [0.] * ref_dict['text_len']
    pred_vec = [0.] * ref_dict['text_len']
    ref_labels = (ref_dict['labels'] if label_filtered_ is None
                  else [s for s in ref_dict['labels'] if s['label'] == label_filtered_])
    pred_labels = (pred_dict['labels'] if label_filtered_ is None
                   else [s for s in pred_dict['labels'] if s['label'] == label_filtered_])
    for span in ref_labels:
        for idx in range(span['start'], span['end']):
            ref_vec[idx] += span['prob']
    for span in pred_labels:
        for idx in range(span['start'], span['end']):
            pred_vec[idx] = span['prob']
    ref_cmps = {round(f, 8) for f in ref_vec}
    pred_cmps = {round(f, 8) for f in pred_vec}
    if len(pred_cmps) == 1 or len(ref_cmps) == 1:
        if len(pred_cmps) != len(ref_cmps):
            return 0.0
        if ref_cmps == {0.0}:
            return float(pred_cmps == {0.0})
        return float(pred_cmps != {0.0})
    return spearmanr(ref_vec, pred_vec).correlation


def score_cor_lbl(ref_dict, pred_dict):
    all_labels = {s['label'] for d in [ref_dict, pred_dict] for s in d['labels']}
    if all_labels:
        return sum(score_cor(ref_dict, pred_dict, label_filtered_=l)
                   for l in all_labels) / len(all_labels)
    return 1.0


def score_iou(ref_dict, pred_dict):
    assert ref_dict['id'] == pred_dict['id']
    ref_i = {i for s in ref_dict['labels'] for i in range(s['start'], s['end'])}
    pred_i = {i for s in pred_dict['labels'] for i in range(s['start'], s['end'])}
    if not pred_i and not ref_i:
        return 1.
    return len(ref_i & pred_i) / len(ref_i | pred_i)


def evaluate(refs, preds):
    refs = sorted(refs, key=lambda r: r['id'])
    preds = sorted(preds, key=lambda r: r['id'])
    assert [r['id'] for r in refs] == [p['id'] for p in preds]
    return {
        'Cor':     float(np.mean([score_cor(r, p)     for r, p in zip(refs, preds)])),
        'Cor_lbl': float(np.mean([score_cor_lbl(r, p) for r, p in zip(refs, preds)])),
        'IoU':     float(np.mean([score_iou(r, p)     for r, p in zip(refs, preds)])),
    }

In [ ]:
# ── Cell 3: data ─────────────────────────────────────────────────────────────
def load(lang, split_name):
    kind = "labeled" if split_name == "train" else "unlabeled"
    path = f"{DISTRIB}/shroom-vision.{split_name}.{lang}.{kind}.jsonl"
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


def split_dev(rows, fraction=0.2, seed=SEED):
    """Stratified split preserving the clean / hallucinated ratio."""
    clean = [r for r in rows if not r["labels"]]
    dirty = [r for r in rows if r["labels"]]
    rng = random.Random(seed)
    rng.shuffle(clean); rng.shuffle(dirty)
    nc, nd = int(len(clean) * fraction), int(len(dirty) * fraction)
    dev = clean[:nc] + dirty[:nd]
    train = clean[nc:] + dirty[nd:]
    rng.shuffle(dev); rng.shuffle(train)
    return train, dev


def char_targets(row):
    n = len(row["response"])
    prob = np.zeros(n, dtype=np.float32)
    cat = np.full(n, -1, dtype=np.int64)
    for span in row.get("labels", []):
        for i in range(span["start"], min(span["end"], n)):
            if span["prob"] >= prob[i]:
                prob[i] = span["prob"]
                cat[i] = CATEGORIES.index(span["label"])
    return prob, cat


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def encode(row):
    """Tokenize response (segment 0) with the prompt as context (segment 1)."""
    enc = tokenizer(row["response"], text_pair=row["prompt"],
                    return_offsets_mapping=True, truncation="only_first",
                    max_length=MAX_LEN)
    seq_ids = enc.sequence_ids()
    keep = [i for i, (a, b) in enumerate(enc["offset_mapping"])
            if b > a and seq_ids[i] == 0]
    return enc, keep


class SpanData(Dataset):
    def __init__(self, rows, labeled=True):
        self.rows, self.labeled = rows, labeled
        # Tokenize once up front. Doing it per access pins the CPU at 100% and
        # starves the GPU, since the loader runs in the main process.
        self.cache = [encode(r) for r in rows]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        enc, keep = self.cache[i]
        item = {
            "input_ids": torch.tensor(enc["input_ids"]),
            "attention_mask": torch.tensor(enc["attention_mask"]),
            "keep": torch.tensor(keep, dtype=torch.long),
        }
        if self.labeled:
            prob, cat = char_targets(row)
            offs = [enc["offset_mapping"][i] for i in keep]
            # per token: max character probability, and the category at the peak
            tp = [float(prob[a:b].max()) if b > a else 0.0 for a, b in offs]
            tc = []
            for a, b in offs:
                seg = cat[a:b]
                seg = seg[seg >= 0]
                tc.append(int(np.bincount(seg).argmax()) if len(seg) else -100)
            item["tok_prob"] = torch.tensor(tp, dtype=torch.float)
            item["tok_cat"] = torch.tensor(tc, dtype=torch.long)
        return item


def collate(batch):
    pad = tokenizer.pad_token_id
    maxlen = max(len(b["input_ids"]) for b in batch)
    maxkeep = max(len(b["keep"]) for b in batch)
    out = {
        "input_ids": torch.full((len(batch), maxlen), pad, dtype=torch.long),
        "attention_mask": torch.zeros((len(batch), maxlen), dtype=torch.long),
        "keep": torch.zeros((len(batch), maxkeep), dtype=torch.long),
        "keep_mask": torch.zeros((len(batch), maxkeep), dtype=torch.bool),
    }
    has_labels = "tok_prob" in batch[0]
    if has_labels:
        out["tok_prob"] = torch.zeros((len(batch), maxkeep))
        out["tok_cat"] = torch.full((len(batch), maxkeep), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L, K = len(b["input_ids"]), len(b["keep"])
        out["input_ids"][i, :L] = b["input_ids"]
        out["attention_mask"][i, :L] = b["attention_mask"]
        out["keep"][i, :K] = b["keep"]
        out["keep_mask"][i, :K] = True
        if has_labels:
            out["tok_prob"][i, :K] = b["tok_prob"]
            out["tok_cat"][i, :K] = b["tok_cat"]
    return out

In [ ]:
# ── Cell 4: model ────────────────────────────────────────────────────────────
class SpanTagger(nn.Module):
    """One shared encoder, two token-level heads.

    span head  -> is this token hallucinated        (drives Cor)
    cat  head  -> which of the five categories      (drives Cor_lbl only)
    """

    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_ID)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.span_head = nn.Linear(h, 1)
        self.cat_head = nn.Linear(h, len(CATEGORIES))

    def forward(self, input_ids, attention_mask, keep, keep_mask):
        hidden = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state
        idx = keep.unsqueeze(-1).expand(-1, -1, hidden.size(-1))
        tok = self.dropout(torch.gather(hidden, 1, idx))
        return self.span_head(tok).squeeze(-1), self.cat_head(tok)


def run_epoch(model, loader, optimizer=None, scheduler=None, log_every=50):
    train = optimizer is not None
    model.train() if train else model.eval()
    bce = nn.BCEWithLogitsLoss(reduction="none")
    ce = nn.CrossEntropyLoss(ignore_index=-100)
    total, nb = 0.0, 0
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.set_grad_enabled(train):
            span_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                          batch["keep"], batch["keep_mask"])
            m = batch["keep_mask"]
            # soft targets: BCE against the annotator probability itself
            l_span = (bce(span_logit, batch["tok_prob"]) * m).sum() / m.sum().clamp(min=1)
            l_cat = ce(cat_logit.reshape(-1, len(CATEGORIES)), batch["tok_cat"].reshape(-1))
            loss = l_span + 0.5 * l_cat
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total += loss.item(); nb += 1
        if train and step % log_every == 0:
            print(f"  step {step}/{len(loader)}  loss {total / nb:.4f}", flush=True)
    return total / max(nb, 1)

In [ ]:
# ── Cell 5: inference ────────────────────────────────────────────────────────
@torch.no_grad()
def predict_char_probs(model, rows):
    """Return per-character probability and category arrays for each row."""
    model.eval()
    dataset = SpanData(rows, labeled=False)
    loader = DataLoader(dataset, batch_size=BATCH, shuffle=False, collate_fn=collate)
    results, cursor = [], 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        span_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                      batch["keep"], batch["keep_mask"])
        probs = torch.sigmoid(span_logit).cpu().numpy()
        cats = cat_logit.argmax(-1).cpu().numpy()
        for i in range(len(probs)):
            row = rows[cursor]
            enc, keep = dataset.cache[cursor]
            cursor += 1
            offs = [enc["offset_mapping"][k] for k in keep]
            n = len(row["response"])
            cp = np.zeros(n, dtype=np.float32)
            cc = np.zeros(n, dtype=np.int64)
            for j, (a, b) in enumerate(offs):
                cp[a:min(b, n)] = probs[i][j]
                cc[a:min(b, n)] = cats[i][j]
            results.append((cp, cc))
    return results


def to_spans(char_prob, char_cat, threshold):
    """Contiguous above-threshold runs, split where the category changes."""
    spans, n, i = [], len(char_prob), 0
    while i < n:
        if char_prob[i] < threshold:
            i += 1; continue
        j, c = i, char_cat[i]
        while j < n and char_prob[j] >= threshold and char_cat[j] == c:
            j += 1
        spans.append({"start": int(i), "end": int(j),
                      "prob": float(round(float(np.mean(char_prob[i:j])), 6)),
                      "label": CATEGORIES[int(c)]})
        i = j
    return spans


def build_preds(rows, char_preds, threshold):
    return [{"id": r["id"], "labels": to_spans(cp, cc, threshold)}
            for r, (cp, cc) in zip(rows, char_preds)]


def tune_threshold(dev_rows, char_preds):
    """Pick the threshold maximizing Cor on dev. Cor is the reported metric."""
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in dev_rows]
    best = (None, -1, None)
    for t in np.arange(0.10, 0.91, 0.05):
        s = evaluate(refs, build_preds(dev_rows, char_preds, float(t)))
        if s["Cor"] > best[1]:
            best = (float(t), s["Cor"], s)
    return best

In [ ]:
# ── Cell 6: train ────────────────────────────────────────────────────────────
train_rows, dev_rows = [], []
for lang in LANGS:
    tr, dv = split_dev(load(lang, "train"))
    train_rows += tr; dev_rows += dv
print(f"train={len(train_rows)}  dev={len(dev_rows)}")

model = SpanTagger().to(DEVICE)

# XLM-R-large's embedding matrix is 256M of its 560M parameters (250k vocab).
# Freezing it drops its gradient and both Adam states — about 3 GB — at little
# cost, since the multilingual embeddings are already well trained.
model.encoder.embeddings.requires_grad_(False)

train_loader = DataLoader(SpanData(train_rows), batch_size=BATCH, shuffle=True,
                          collate_fn=collate)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01, foreach=False)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(0.1 * len(train_loader) * EPOCHS), len(train_loader) * EPOCHS)

for epoch in range(EPOCHS):
    loss = run_epoch(model, train_loader, optimizer, scheduler)
    print(f"epoch {epoch + 1}: loss {loss:.4f}", flush=True)
    # checkpoint every epoch so a disconnect costs one epoch, not the whole run
    torch.save(model.state_dict(), f"{OUT_DIR}/span_tagger.pt")

In [ ]:
# ── Cell 7: evaluate per language ────────────────────────────────────────────
print(f"\n{'lang':<6}{'thr':>6}{'Cor':>9}{'Cor_lbl':>9}{'IoU':>9}   (mark_none Cor)")
thresholds = {}
for lang in LANGS:
    rows = [r for r in dev_rows if r["language"] == lang]
    cps = predict_char_probs(model, rows)
    thr, cor, scores = tune_threshold(rows, cps)
    thresholds[lang] = thr
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in rows]
    floor = evaluate(refs, [{"id": r["id"], "labels": []} for r in rows])["Cor"]
    print(f"{lang:<6}{thr:>6.2f}{scores['Cor']:>9.3f}{scores['Cor_lbl']:>9.3f}"
          f"{scores['IoU']:>9.3f}   {floor:.3f}")


In [ ]:
# ── Cell 8: predict test and write submission ────────────────────────────────
for lang in LANGS:
    rows = load(lang, "test")
    cps = predict_char_probs(model, rows)
    preds = build_preds(rows, cps, thresholds[lang])
    path = f"{OUT_DIR}/predictions_{lang}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for p in preds:
            fh.write(json.dumps(p, ensure_ascii=False) + "\n")
    n_spans = sum(len(p["labels"]) for p in preds)
    n_empty = sum(1 for p in preds if not p["labels"])
    print(f"{lang}: {len(preds)} rows, {n_spans} spans, {n_empty} empty -> {path}")


In [ ]:
!wget -q https://a3s.fi/mickusti-2007780-pub/participant_kit.shroom_visions.zip -O /tmp/kit.zip
!unzip -oq /tmp/kit.zip -d /tmp/kit
!cd /kaggle/working && python /tmp/kit/participant_kit/format_checker.py predictions_en.jsonl predictions_fr.jsonl predictions_it.jsonl predictions_zh.jsonl --reference-dir /kaggle/working/distrib

In [ ]:
from IPython.display import FileLink
!cd /kaggle/working && zip -q predictions.zip predictions_en.jsonl predictions_fr.jsonl predictions_it.jsonl predictions_zh.jsonl
FileLink('predictions.zip')

In [ ]:
# ── Cell 9: category assignment strategies (no retraining needed) ────────────
#
# The span head stays exactly as trained. Only the way we turn per-token
# category scores into span labels changes. Compares three strategies on dev.

import numpy as np
import torch
from torch.utils.data import DataLoader


@torch.no_grad()
def predict_char_full(model, rows):
    """Per-character hallucination probability plus the full category distribution.

    Returns (char_prob, char_cat_probs) per row, where char_cat_probs has shape
    (n_chars, len(CATEGORIES)). Keeping the distribution instead of an argmax is
    what lets us pool categories over a span or a whole response.
    """
    model.eval()
    dataset = SpanData(rows, labeled=False)
    loader = DataLoader(dataset, batch_size=BATCH, shuffle=False, collate_fn=collate)
    results, cursor = [], 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        span_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                      batch["keep"], batch["keep_mask"])
        probs = torch.sigmoid(span_logit).cpu().numpy()
        cat_p = torch.softmax(cat_logit, dim=-1).cpu().numpy()
        for i in range(len(probs)):
            row = rows[cursor]
            enc, keep = dataset.cache[cursor]
            cursor += 1
            offs = [enc["offset_mapping"][k] for k in keep]
            n = len(row["response"])
            cp = np.zeros(n, dtype=np.float32)
            cd = np.zeros((n, len(CATEGORIES)), dtype=np.float32)
            for j, (a, b) in enumerate(offs):
                e = min(b, n)
                cp[a:e] = probs[i][j]
                cd[a:e] = cat_p[i][j]
            results.append((cp, cd))
    return results


def spans_with_strategy(char_prob, char_dist, threshold, strategy):
    """Build spans, assigning categories by one of three strategies.

    per_token : argmax at each character; a span breaks where the label changes
    per_span  : one label per contiguous run, pooled over its characters
    per_resp  : one label for the whole response, pooled over flagged characters
    """
    n = len(char_prob)
    above = char_prob >= threshold
    if not above.any():
        return []

    resp_label = int(char_dist[above].sum(axis=0).argmax())

    # contiguous runs of flagged characters
    runs, i = [], 0
    while i < n:
        if not above[i]:
            i += 1
            continue
        j = i
        while j < n and above[j]:
            j += 1
        runs.append((i, j))
        i = j

    spans = []
    for a, b in runs:
        if strategy == "per_resp":
            pieces = [(a, b, resp_label)]
        elif strategy == "per_span":
            pieces = [(a, b, int(char_dist[a:b].sum(axis=0).argmax()))]
        else:  # per_token — split the run wherever the argmax label changes
            labels = char_dist[a:b].argmax(axis=1)
            pieces, s = [], 0
            for k in range(1, len(labels) + 1):
                if k == len(labels) or labels[k] != labels[s]:
                    pieces.append((a + s, a + k, int(labels[s])))
                    s = k
        for s, e, lab in pieces:
            spans.append({"start": int(s), "end": int(e),
                          "prob": float(round(float(char_prob[s:e].mean()), 6)),
                          "label": CATEGORIES[lab]})
    return spans


def eval_strategy(rows, char_preds, threshold, strategy):
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in rows]
    preds = [{"id": r["id"],
              "labels": spans_with_strategy(cp, cd, threshold, strategy)}
             for r, (cp, cd) in zip(rows, char_preds)]
    scores = evaluate(refs, preds)
    counts = [len({s["label"] for s in p["labels"]}) for p in preds if p["labels"]]
    scores["cats_per_resp"] = float(np.mean(counts)) if counts else 0.0
    scores["n_flagged"] = len(counts)
    return scores


# ── run the comparison ───────────────────────────────────────────────────────
STRATEGIES = ["per_token", "per_span", "per_resp"]
best_strategy, best_thr = {}, {}

print(f"{'lang':<5}{'strategy':<12}{'thr':>6}{'Cor':>9}{'Cor_lbl':>10}{'cats/resp':>11}")
print("-" * 53)
for lang in LANGS:
    rows = [r for r in dev_rows if r["language"] == lang]
    preds_full = predict_char_full(model, rows)
    best = (None, None, -1)
    for strategy in STRATEGIES:
        top = (None, -1, None)
        for t in np.arange(0.10, 0.91, 0.05):
            s = eval_strategy(rows, preds_full, float(t), strategy)
            # Cor+Lbl is listed first on the leaderboard, so tune for it
            if s["Cor_lbl"] > top[1]:
                top = (float(t), s["Cor_lbl"], s)
        thr, cl, s = top
        print(f"{lang:<5}{strategy:<12}{thr:>6.2f}{s['Cor']:>9.3f}"
              f"{s['Cor_lbl']:>10.3f}{s['cats_per_resp']:>11.2f}")
        if cl > best[2]:
            best = (strategy, thr, cl)
    best_strategy[lang], best_thr[lang] = best[0], best[1]
    print(f"{'':5}-> best: {best[0]} @ {best[1]:.2f}  (Cor_lbl {best[2]:.3f})")
    print()

print("chosen:", {l: (best_strategy[l], round(best_thr[l], 2)) for l in LANGS})

In [ ]:
!ls /kaggle/working/shroom-vis-images | head -3
!ls /kaggle/working/shroom-vis-images | wc -l

In [ ]:
# =============================================================================
#  SHROOM-visions 2026 — Day 4: visual grounding features
#
#  For every response token, score it twice under Qwen2-VL-2B with teacher
#  forcing: once WITH the image, once with the image REMOVED. The difference in
#  log-probability measures how much that token depended on seeing the image.
#
#  Removal, not Gaussian noise: Yin et al. (arXiv 2504.10020) show noise makes
#  the model randomly overlook parts of the image, so the gap reflects what the
#  noise destroyed rather than what was grounded. Full removal is deterministic.
#
#  Output: visual_feats_{split}_{lang}.npz  — per example, character offsets
#  plus six per-token features.
# =============================================================================

# ── Cell V1: setup ───────────────────────────────────────────────────────────
SMOKE_TEST = False
MAX_EXAMPLES = 300

import json, os, time, gc
import numpy as np
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

DISTRIB   = "/kaggle/working/distrib"
IMAGE_DIR = "/kaggle/working/shroom-vis-images"
OUT_DIR   = "/kaggle/working"
MODEL_ID  = "Qwen/Qwen2-VL-2B-Instruct"
LANGS     = ["en", "fr", "it", "zh"]

# Cap visual tokens. Qwen2-VL uses dynamic resolution, and an unconstrained
# photo can produce >1500 visual tokens — the difference between a 3-hour run
# and a 12-hour one.
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 256 * 28 * 28

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


# ── Cell V2: load model ──────────────────────────────────────────────────────
# 2B in fp16 is ~4.4 GB — no quantization needed on a 15 GB T4, and fp16
# inference is faster than 4-bit at this size.
try:
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID, dtype=torch.float16, device_map="auto")
except TypeError:  # older transformers spells it torch_dtype
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
tokenizer = processor.tokenizer
print("model loaded")


# ── Cell V3: scoring ─────────────────────────────────────────────────────────
@torch.no_grad()
def score_response(image, prompt, response):
    """Teacher-force `response` twice and return per-token features.

    Returns (offsets, feats) where feats has one row per response token:
        [logp_img, ent_img, logp_noimg, ent_noimg, d_logp, d_ent]
    or (None, None) if the two tokenizations cannot be aligned.
    """
    # Character offsets come from tokenizing the response on its own.
    resp_enc = tokenizer(response, return_offsets_mapping=True,
                         add_special_tokens=False)
    resp_ids = resp_enc["input_ids"]
    offsets = resp_enc["offset_mapping"]
    if not resp_ids:
        return None, None

    def run(with_image):
        content = ([{"type": "image"}] if with_image else []) + \
                  [{"type": "text", "text": prompt}]
        messages = [{"role": "user", "content": content}]
        prefix = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        kwargs = {"text": [prefix + response], "return_tensors": "pt"}
        pre_kwargs = {"text": [prefix], "return_tensors": "pt"}
        if with_image:
            kwargs["images"] = [image]
            pre_kwargs["images"] = [image]
        inputs = processor(**kwargs).to(DEVICE)
        n_prefix = processor(**pre_kwargs)["input_ids"].shape[1]

        ids = inputs["input_ids"][0]
        # The response must tokenize identically in context, or offsets are wrong.
        if len(ids) - n_prefix != len(resp_ids):
            return None
        logits = model(**inputs).logits[0].float()
        # position i predicts token i+1
        step = logits[n_prefix - 1:-1]
        logprobs = torch.log_softmax(step, dim=-1)
        target = ids[n_prefix:]
        tok_logp = logprobs.gather(1, target.unsqueeze(1)).squeeze(1)
        entropy = -(logprobs.exp() * logprobs).sum(dim=-1)
        return tok_logp.cpu().numpy(), entropy.cpu().numpy()

    with_img = run(True)
    without_img = run(False)
    if with_img is None or without_img is None:
        return None, None

    logp_i, ent_i = with_img
    logp_n, ent_n = without_img
    feats = np.stack([logp_i, ent_i, logp_n, ent_n,
                      logp_i - logp_n, ent_i - ent_n], axis=1)
    return np.array(offsets, dtype=np.int32), feats.astype(np.float32)


# ── Cell V4: run over a split ────────────────────────────────────────────────
def load_rows(split, lang):
    kind = "labeled" if split == "train" else "unlabeled"
    with open(f"{DISTRIB}/shroom-vision.{split}.{lang}.{kind}.jsonl",
              encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


def extract(split, lang, limit=None):
    rows = load_rows(split, lang)
    if limit:
        rows = rows[:limit]
    out_path = f"{OUT_DIR}/visual_feats_{split}_{lang}.npz"

    store, skipped, t0 = {}, 0, time.time()
    for i, row in enumerate(rows):
        img_path = os.path.join(IMAGE_DIR, row["image_name"])
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as exc:
            skipped += 1
            if skipped <= 3:
                print(f"  image unreadable {row['image_name']}: {exc}")
            continue
        offsets, feats = score_response(image, row["prompt"], row["response"])
        if feats is None:
            skipped += 1
            continue
        store[row["id"] + "|off"] = offsets
        store[row["id"] + "|f"] = feats
        if (i + 1) % 100 == 0:
            rate = (i + 1) / (time.time() - t0)
            eta = (len(rows) - i - 1) / rate / 60
            print(f"  {split}/{lang} {i + 1}/{len(rows)} "
                  f"{rate:.1f}/s  eta {eta:.0f} min  skipped {skipped}",
                  flush=True)

    np.savez_compressed(out_path, **store)
    done = len(store) // 2
    print(f"{split}/{lang}: {done} saved, {skipped} skipped, "
          f"{(time.time() - t0) / 60:.1f} min -> {out_path}", flush=True)
    return done, skipped

# ── Cell V5: run extraction ──────────────────────────────────────────────────
total = 0
for split in ["train"]:
    for lang in ["en"]:
        done, _ = extract(split, lang, MAX_EXAMPLES)
        total += done
        gc.collect(); torch.cuda.empty_cache()
print(f"\nall done: {total} examples")

In [ ]:
# ── Cell V6: does the signal exist? ──────────────────────────────────────────
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

d = np.load(f"{OUT_DIR}/visual_feats_train_en.npz")
rows = {json.loads(l)["id"]: json.loads(l)
        for l in open(f"{DISTRIB}/shroom-vision.train.en.labeled.jsonl")}

ys, gs, xs = [], [], []
for key in [k for k in d.files if k.endswith("|f")]:
    rid = key[:-2]
    row, offs, feats = rows[rid], d[rid + "|off"], d[key]
    gold = np.zeros(len(row["response"]))
    for s in row["labels"]:
        seg = gold[s["start"]:s["end"]]
        gold[s["start"]:s["end"]] = np.maximum(seg, s["prob"])
    for (a, b), f in zip(offs, feats):
        g = float(gold[a:b].max()) if b > a else 0.0
        ys.append(g > 0)        # any annotator marked it
        gs.append(g)            # graded probability, what Cor actually uses
        xs.append(f)

ys, gs, xs = np.array(ys), np.array(gs), np.array(xs)
names = ["logp_img", "ent_img", "logp_noimg", "ent_noimg", "d_logp", "d_ent"]
print(f"{ys.sum()} hallucinated / {len(ys)} tokens ({100*ys.mean():.1f}%)\n")
print(f"{'feature':<12}{'AUC':>8}{'Spearman':>11}")
for i, n in enumerate(names):
    auc = roc_auc_score(ys, -xs[:, i])
    rho = spearmanr(gs, -xs[:, i]).correlation
    print(f"{n:<12}{auc:>8.3f}{rho:>11.3f}")

In [ ]:
# ── Cell V7: are the features actually aligned to their tokens? ──────────────
from collections import defaultdict
by_tok = defaultdict(list)
for key in [k for k in d.files if k.endswith("|f")]:
    rid = key[:-2]
    resp, offs, feats = rows[rid]["response"], d[rid + "|off"], d[key]
    for (a, b), f in zip(offs, feats):
        by_tok[resp[a:b]].append(f[0])          # logp_img

common = {t: np.mean(v) for t, v in by_tok.items() if len(v) >= 20}
order = sorted(common, key=common.get)
print("LOWEST logp (should be rare/contentful):", [repr(t) for t in order[:10]])
print("HIGHEST logp (should be common/function):", [repr(t) for t in order[-10:]])

# Visual Ablation Probe

Same two forward passes, different thing recorded. Instead of output log-probabilities we take **internal representations**, because the proxy-analyzer literature ([arXiv 2605.07209](https://arxiv.org/html/2605.07209)) gets cross-model transfer from activations, while our logit features sat at chance.

Per token, from three layers (early / middle / late), with and without the image:

| feature | question it asks |
|---|---|
| `cos(h_img, h_noimg)` | did the image rotate this token's representation? |
| `‖h_img − h_noimg‖` | by how much? |

Plus a fixed 32-dimensional random projection of the middle-layer difference, so a classifier can look for structure a single scalar would miss.

In [ ]:
# ── Cell H1: hidden-state features ───────────────────────────────────────────

import numpy as np, torch, json, os, time, gc

PROJ_DIM = 32
_rng = np.random.RandomState(0)
_proj = None   # built lazily once the hidden size is known


@torch.no_grad()
def score_response_hidden(image, prompt, response):
    global _proj
    resp_enc = tokenizer(response, return_offsets_mapping=True,
                         add_special_tokens=False)
    resp_ids = resp_enc["input_ids"]
    offsets = resp_enc["offset_mapping"]
    if not resp_ids:
        return None, None
    n_resp = len(resp_ids)

    def run(with_image):
        content = ([{"type": "image"}] if with_image else []) + \
                  [{"type": "text", "text": prompt}]
        messages = [{"role": "user", "content": content}]
        prefix = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        kwargs = {"text": [prefix + response], "return_tensors": "pt"}
        if with_image:
            kwargs["images"] = [image]
        inputs = processor(**kwargs).to(DEVICE)
        out = model(**inputs, output_hidden_states=True)
        hs = out.hidden_states
        n = len(hs)
        picks = [n // 4, n // 2, (3 * n) // 4]
        # response tokens are the final n_resp positions in both passes
        return [hs[p][0, -n_resp:, :].float().cpu().numpy() for p in picks]

    a = run(True)
    b = run(False)
    if a[0].shape[0] != n_resp or b[0].shape[0] != n_resp:
        return None, None

    if _proj is None:
        _proj = _rng.randn(a[1].shape[1], PROJ_DIM).astype(np.float32) / np.sqrt(PROJ_DIM)

    cols = []
    for hi, hn in zip(a, b):
        num = (hi * hn).sum(axis=1)
        den = np.linalg.norm(hi, axis=1) * np.linalg.norm(hn, axis=1) + 1e-8
        cols.append(num / den)                        # cosine
        cols.append(np.linalg.norm(hi - hn, axis=1))  # distance
    feats = np.stack(cols, axis=1)
    proj = (a[1] - b[1]) @ _proj
    return np.array(offsets, dtype=np.int32), \
        np.concatenate([feats, proj], axis=1).astype(np.float32)

In [ ]:
# ── Cell H2: extract ─────────────────────────────────────────────────────────
def extract_hidden(split, lang, limit=None):
    rows = load_rows(split, lang)
    if limit:
        rows = rows[:limit]
    store, skipped, t0 = {}, 0, time.time()
    for i, row in enumerate(rows):
        try:
            image = Image.open(os.path.join(IMAGE_DIR, row["image_name"])).convert("RGB")
        except Exception:
            skipped += 1
            continue
        offsets, feats = score_response_hidden(image, row["prompt"], row["response"])
        if feats is None:
            skipped += 1
            continue
        store[row["id"] + "|off"] = offsets
        store[row["id"] + "|f"] = feats
        if (i + 1) % 100 == 0:
            rate = (i + 1) / (time.time() - t0)
            print(f"  {i + 1}/{len(rows)} {rate:.1f}/s skipped {skipped}", flush=True)
    path = f"{OUT_DIR}/hidden_feats_{split}_{lang}.npz"
    np.savez_compressed(path, **store)
    print(f"{len(store) // 2} saved, {skipped} skipped, "
          f"{(time.time() - t0) / 60:.1f} min -> {path}", flush=True)


for split in ["test", "train"]:
    for lang in ["en", "fr", "it", "zh"]:
        print(f"=== {split}/{lang} ===", flush=True)
        extract_hidden(split, lang, None)
        gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ── Cell H3: is there signal? ────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr

d = np.load(f"{OUT_DIR}/hidden_feats_train_en.npz")
rows = {json.loads(l)["id"]: json.loads(l)
        for l in open(f"{DISTRIB}/shroom-vision.train.en.labeled.jsonl")}

ys, gs, xs = [], [], []
for key in [k for k in d.files if k.endswith("|f")]:
    rid = key[:-2]
    row, offs, feats = rows[rid], d[rid + "|off"], d[key]
    gold = np.zeros(len(row["response"]))
    for s in row["labels"]:
        seg = gold[s["start"]:s["end"]]
        gold[s["start"]:s["end"]] = np.maximum(seg, s["prob"])
    for (a, b), f in zip(offs, feats):
        g = float(gold[a:b].max()) if b > a else 0.0
        ys.append(g > 0); gs.append(g); xs.append(f)

ys, gs, xs = np.array(ys), np.array(gs), np.array(xs)
names = ["cos_early", "dist_early", "cos_mid", "dist_mid", "cos_late", "dist_late"]
print(f"{ys.sum()} hallucinated / {len(ys)} tokens ({100 * ys.mean():.1f}%)\n")
print(f"{'feature':<12}{'AUC':>8}{'Spearman':>11}")
for i, n in enumerate(names):
    # try both directions; a scalar can separate either way
    auc = roc_auc_score(ys, xs[:, i])
    print(f"{n:<12}{max(auc, 1 - auc):>8.3f}{spearmanr(gs, xs[:, i]).correlation:>11.3f}")

# can a classifier find structure in the full difference vector?
Xtr, Xte, ytr, yte = train_test_split(xs, ys, test_size=0.3, random_state=0, stratify=ys)
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit((Xtr - mu) / sd, ytr)
probe = roc_auc_score(yte, clf.predict_proba((Xte - mu) / sd)[:, 1])
print(f"\nlogistic probe on all {xs.shape[1]} features: AUC {probe:.3f}")
print("(0.50 = no signal; >0.60 means activations carry what logits did not)")

In [ ]:
# ── Cell H4: same probe, split by response instead of by token ───────────────
from sklearn.model_selection import GroupShuffleSplit

groups = []
ys2, xs2 = [], []
for key in [k for k in d.files if k.endswith("|f")]:
    rid = key[:-2]
    row, offs, feats = rows[rid], d[rid + "|off"], d[key]
    gold = np.zeros(len(row["response"]))
    for s in row["labels"]:
        seg = gold[s["start"]:s["end"]]
        gold[s["start"]:s["end"]] = np.maximum(seg, s["prob"])
    for (a, b), f in zip(offs, feats):
        ys2.append((float(gold[a:b].max()) if b > a else 0.0) > 0)
        xs2.append(f); groups.append(rid)

ys2, xs2, groups = np.array(ys2), np.array(xs2), np.array(groups)
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
              .split(xs2, ys2, groups))
mu, sd = xs2[tr].mean(0), xs2[tr].std(0) + 1e-8
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit((xs2[tr] - mu) / sd, ys2[tr])
auc = roc_auc_score(ys2[te], clf.predict_proba((xs2[te] - mu) / sd)[:, 1])
print(f"grouped by response: AUC {auc:.3f}   ({len(set(groups))} responses)")

# which half is doing the work — the 6 scalars or the 32 projection dims?
for label, cols in [("6 scalars only", slice(0, 6)), ("32 projection only", slice(6, 38))]:
    m, s = xs2[tr][:, cols].mean(0), xs2[tr][:, cols].std(0) + 1e-8
    c = LogisticRegression(max_iter=2000, class_weight="balanced")
    c.fit((xs2[tr][:, cols] - m) / s, ys2[tr])
    a = roc_auc_score(ys2[te], c.predict_proba((xs2[te][:, cols] - m) / s)[:, 1])
    print(f"  {label:<20} AUC {a:.3f}")

In [ ]:
# ── FUSED TAGGER: XLM-R + visual ablation features ───────────────────────────
VIS_DIM   = 38
VIS_PROJ  = 64
USE_VIS   = True        # flip to False later for the text-only control

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, get_linear_schedule_with_warmup

# --- load visual features, mapped onto characters ---------------------------
def load_visual(split, lang):
    path = f"{OUT_DIR}/hidden_feats_{split}_{lang}.npz"
    if not os.path.exists(path):
        print(f"  missing {path}"); return {}
    d = np.load(path); out = {}
    for key in d.files:
        if not key.endswith("|f"): continue
        rid = key[:-2]
        offs, feats = d[rid + "|off"], d[key]
        n = int(offs[-1][1]) if len(offs) else 0
        arr = np.zeros((n, VIS_DIM), dtype=np.float32)
        for (a, b), f in zip(offs, feats):
            arr[a:b] = f
        out[rid] = arr
    return out

VISUAL = {}
for split in ["train", "test"]:
    for lang in LANGS:
        VISUAL.update(load_visual(split, lang))
print(f"visual features for {len(VISUAL)} examples")

# Floor the standard deviation. cos_late is nearly constant (AUC 0.503), so a
# 1e-6 guard divides by ~zero and blows the features up into NaN territory.
_sample = np.concatenate([v for k, v in list(VISUAL.items())[:2000]], axis=0)
VIS_MEAN, VIS_STD = _sample.mean(axis=0), np.maximum(_sample.std(axis=0), 1e-2)

def visual_for_tokens(rid, offsets):
    arr = VISUAL.get(rid)
    out = np.zeros((len(offsets), VIS_DIM), dtype=np.float32)
    if arr is None:
        return out, 0.0
    for i, (a, b) in enumerate(offsets):
        b = min(b, len(arr))
        if b > a:
            out[i] = arr[a:b].mean(axis=0)
    return (out - VIS_MEAN) / VIS_STD, 1.0

# --- dataset ----------------------------------------------------------------
class FusedData(Dataset):
    def __init__(self, rows, labeled=True):
        self.rows, self.labeled = rows, labeled
        self.cache = [encode(r) for r in rows]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        row = self.rows[i]
        enc, keep = self.cache[i]
        offs = [enc["offset_mapping"][k] for k in keep]
        vis, has_vis = visual_for_tokens(row["id"], offs)
        item = {
            "input_ids": torch.tensor(enc["input_ids"]),
            "attention_mask": torch.tensor(enc["attention_mask"]),
            "keep": torch.tensor(keep, dtype=torch.long),
            "vis": torch.tensor(vis),
            "has_vis": torch.tensor(has_vis),
        }
        if self.labeled:
            prob, cat = char_targets(row)
            tp = [float(prob[a:b].max()) if b > a else 0.0 for a, b in offs]
            tc = []
            for a, b in offs:
                seg = cat[a:b]; seg = seg[seg >= 0]
                tc.append(int(np.bincount(seg).argmax()) if len(seg) else -100)
            item["tok_prob"] = torch.tensor(tp, dtype=torch.float)
            item["tok_cat"] = torch.tensor(tc, dtype=torch.long)
        return item

def collate_fused(batch):
    pad = tokenizer.pad_token_id
    maxlen = max(len(b["input_ids"]) for b in batch)
    maxkeep = max(len(b["keep"]) for b in batch)
    n = len(batch)
    out = {
        "input_ids": torch.full((n, maxlen), pad, dtype=torch.long),
        "attention_mask": torch.zeros((n, maxlen), dtype=torch.long),
        "keep": torch.zeros((n, maxkeep), dtype=torch.long),
        "keep_mask": torch.zeros((n, maxkeep), dtype=torch.bool),
        "vis": torch.zeros((n, maxkeep, VIS_DIM)),
        "has_vis": torch.stack([b["has_vis"] for b in batch]),
    }
    has_labels = "tok_prob" in batch[0]
    if has_labels:
        out["tok_prob"] = torch.zeros((n, maxkeep))
        out["tok_cat"] = torch.full((n, maxkeep), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L, K = len(b["input_ids"]), len(b["keep"])
        out["input_ids"][i, :L] = b["input_ids"]
        out["attention_mask"][i, :L] = b["attention_mask"]
        out["keep"][i, :K] = b["keep"]
        out["keep_mask"][i, :K] = True
        out["vis"][i, :K] = b["vis"]
        if has_labels:
            out["tok_prob"][i, :K] = b["tok_prob"]
            out["tok_cat"][i, :K] = b["tok_cat"]
    return out

# --- model ------------------------------------------------------------------
class FusedTagger(nn.Module):
    def __init__(self, use_vis=USE_VIS):
        super().__init__()
        self.use_vis = use_vis
        self.encoder = AutoModel.from_pretrained(MODEL_ID)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        if use_vis:
            self.vis_proj = nn.Sequential(
                nn.Linear(VIS_DIM, VIS_PROJ), nn.GELU(), nn.LayerNorm(VIS_PROJ))
            h = h + VIS_PROJ
        self.span_head = nn.Linear(h, 1)
        self.cat_head = nn.Linear(h, len(CATEGORIES))
    def forward(self, input_ids, attention_mask, keep, keep_mask, vis, has_vis):
        hidden = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state
        idx = keep.unsqueeze(-1).expand(-1, -1, hidden.size(-1))
        tok = torch.gather(hidden, 1, idx)
        if self.use_vis:
            tok = torch.cat([tok, self.vis_proj(vis) * has_vis.view(-1, 1, 1)], dim=-1)
        tok = self.dropout(tok)
        return self.span_head(tok).squeeze(-1), self.cat_head(tok)

def run_epoch_fused(model, loader, optimizer=None, scheduler=None, log_every=50):
    train = optimizer is not None
    model.train() if train else model.eval()
    bce = nn.BCEWithLogitsLoss(reduction="none")
    ce = nn.CrossEntropyLoss(ignore_index=-100)
    total, nb = 0.0, 0
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.set_grad_enabled(train):
            span_logit, cat_logit = model(
                batch["input_ids"], batch["attention_mask"], batch["keep"],
                batch["keep_mask"], batch["vis"], batch["has_vis"])
            m = batch["keep_mask"]
            l_span = (bce(span_logit, batch["tok_prob"]) * m).sum() / m.sum().clamp(min=1)
            # cross-entropy returns NaN when every target in the batch is ignored,
            # which happens when a batch draws only clean responses
            valid = (batch["tok_cat"] != -100).any()
            l_cat = (ce(cat_logit.reshape(-1, len(CATEGORIES)),
                        batch["tok_cat"].reshape(-1)) if valid
                     else span_logit.sum() * 0.0)
            loss = l_span + 0.5 * l_cat
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total += loss.item(); nb += 1
        if train and step % log_every == 0:
            print(f"  step {step}/{len(loader)}  loss {total / nb:.4f}", flush=True)
    return total / max(nb, 1)

@torch.no_grad()
def predict_fused(model, rows):
    model.eval()
    ds = FusedData(rows, labeled=False)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False, collate_fn=collate_fused)
    results, cursor = [], 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        span_logit, cat_logit = model(
            batch["input_ids"], batch["attention_mask"], batch["keep"],
            batch["keep_mask"], batch["vis"], batch["has_vis"])
        probs = torch.sigmoid(span_logit).cpu().numpy()
        cats = cat_logit.argmax(-1).cpu().numpy()
        for i in range(len(probs)):
            row = rows[cursor]
            enc, keep = ds.cache[cursor]
            cursor += 1
            offs = [enc["offset_mapping"][k] for k in keep]
            n = len(row["response"])
            cp = np.zeros(n, dtype=np.float32)
            cc = np.zeros(n, dtype=np.int64)
            for j, (a, b) in enumerate(offs):
                cp[a:min(b, n)] = probs[i][j]
                cc[a:min(b, n)] = cats[i][j]
            results.append((cp, cc))
    return results

# --- train and evaluate -----------------------------------------------------
train_rows, dev_rows = [], []
for lang in LANGS:
    tr, dv = split_dev(load(lang, "train"))
    train_rows += tr; dev_rows += dv
print(f"train={len(train_rows)}  dev={len(dev_rows)}")

model = FusedTagger(USE_VIS).to(DEVICE)
if MODEL_ID.endswith("large"):
    model.encoder.embeddings.requires_grad_(False)

loader = DataLoader(FusedData(train_rows), batch_size=BATCH, shuffle=True,
                    collate_fn=collate_fused)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=LR, weight_decay=0.01, foreach=False)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(0.1 * len(loader) * EPOCHS), len(loader) * EPOCHS)

for epoch in range(EPOCHS):
    loss = run_epoch_fused(model, loader, optimizer, scheduler)
    print(f"epoch {epoch + 1}: loss {loss:.4f}", flush=True)
    torch.save(model.state_dict(), f"{OUT_DIR}/fused_tagger_vis{int(USE_VIS)}.pt")

print(f"\n{'lang':<6}{'thr':>6}{'Cor':>9}{'Cor_lbl':>9}{'IoU':>9}   (floor)")
thresholds = {}
for lang in LANGS:
    rows = [r for r in dev_rows if r["language"] == lang]
    cps = predict_fused(model, rows)
    thr, cor, scores = tune_threshold(rows, cps)
    thresholds[lang] = thr
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in rows]
    floor = evaluate(refs, [{"id": r["id"], "labels": []} for r in rows])["Cor"]
    print(f"{lang:<6}{thr:>6.2f}{scores['Cor']:>9.3f}{scores['Cor_lbl']:>9.3f}"
          f"{scores['IoU']:>9.3f}   {floor:.3f}")

# --- write submission -------------------------------------------------------
for lang in LANGS:
    rows = load(lang, "test")
    cps = predict_fused(model, rows)
    preds = build_preds(rows, cps, thresholds[lang])
    path = f"{OUT_DIR}/predictions_vis_{lang}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for p in preds:
            fh.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"{lang}: {len(preds)} rows, {sum(len(p['labels']) for p in preds)} spans -> {path}")

In [ ]:
lang = "zh"
rows = load(lang, "test")
cps = predict_fused(model, rows)
preds = build_preds(rows, cps, thresholds[lang])
path = f"{OUT_DIR}/predictions_vis_{lang}.jsonl"
with open(path, "w", encoding="utf-8") as fh:
    for p in preds:
        fh.write(json.dumps(p, ensure_ascii=False) + "\n")
!ls -la /kaggle/working/predictions_vis_zh.jsonl

In [ ]:
!cd /kaggle/working && rm -f preds_vis.zip && zip -q preds_vis.zip predictions_vis_*.jsonl && unzip -l preds_vis.zip

In [ ]:
!cd /kaggle/working && rm -rf __MACOSX "=0.46.1" shroom_data.zip shroom_data.ziporut0_9w.part predictions.zip preds_vis.zip predictions.jsonl && df -h /kaggle/working

In [ ]:
# ── Cell G1: caption every unique image with gemma4 via the WiLine endpoint ──
#
# Runs on Kaggle (where the images are) and calls the endpoint over the network,
# so it costs no GPU quota.
#
# The captions are QUESTION-CONDITIONED. 42% of the dataset's questions are
# fine-grained attribute checks ("does this grater have a rectangular blade?"),
# and a generic caption is simply silent on those — useless as evidence. Since
# each image carries only ~4 distinct questions across all four languages, one
# call per image can address all of them, so this costs no more than a generic
# caption would.
#
# Put the key in Kaggle Secrets first — never in a cell:
#   Add-ons -> Secrets -> add WEC_API_KEY -> attach to this notebook.

SMOKE = False          # caption 5 and stop; set False for the full run
WORKERS = 8
MAX_QUESTIONS = 8     # a few images carry up to 17; keep the prompt bounded

import base64, io, json, os, time, urllib.request, urllib.error
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image

DISTRIB   = "/kaggle/working/distrib"
IMAGE_DIR = "/kaggle/working/shroom-vis-images"
OUT_DIR   = "/kaggle/working"
LANGS     = ["en", "fr", "it", "zh"]

BASE = "https://inference.wiline.com/v1/chat/completions"
CAPTION_MODEL = "gemma4"
CAPTION_PATH = f"{OUT_DIR}/captions_gemma4.json"

from kaggle_secrets import UserSecretsClient
API_KEY = UserSecretsClient().get_secret("WEC_API_KEY")


def load_rows(split, lang):
    kind = "labeled" if split == "train" else "unlabeled"
    with open(f"{DISTRIB}/shroom-vision.{split}.{lang}.{kind}.jsonl",
              encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


def questions_by_image():
    """image -> the distinct questions asked about it, across every split."""
    out = defaultdict(list)
    for split in ["train", "test"]:
        for lang in LANGS:
            for row in load_rows(split, lang):
                q = row["prompt"].strip()
                if q not in out[row["image_name"]]:
                    out[row["image_name"]].append(q)
    return dict(out)


QUESTIONS = questions_by_image()
print(f"{len(QUESTIONS)} unique images, "
      f"{sum(len(v) for v in QUESTIONS.values())} image-question pairs")


def build_prompt(questions):
    listed = "\n".join(f"{i}. {q}" for i, q in enumerate(questions[:MAX_QUESTIONS], 1))
    return (
        "Describe this image factually and in detail. State the objects present, "
        "their colours, shapes and other visible attributes, how many of each "
        "there are, where they are relative to each other, and transcribe any "
        "text visible in the image.\n\n"
        "Your description must explicitly answer each of these questions:\n"
        f"{listed}\n\n"
        "Describe only what you can actually see. If something cannot be "
        "determined from the image, say so plainly rather than guessing."
    )


def _encode(name, max_side=1024):
    img = Image.open(os.path.join(IMAGE_DIR, name)).convert("RGB")
    img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()


def caption_one(name):
    body = json.dumps({
        "model": CAPTION_MODEL,
        "max_tokens": 700,
        "think": False,
        "num_ctx": 8192,
        "temperature": 0,
        "messages": [{"role": "user", "content": [
            {"type": "text", "text": build_prompt(QUESTIONS.get(name, []))},
            {"type": "image_url", "image_url": {
                "url": f"data:image/jpeg;base64,{_encode(name)}"}}]}],
    }).encode()
    for attempt in range(4):
        try:
            req = urllib.request.Request(
                BASE, data=body,
                headers={"Authorization": f"Bearer {API_KEY}",
                         "Content-Type": "application/json"})
            with urllib.request.urlopen(req, timeout=240) as r:
                msg = json.load(r)["choices"][0]["message"]["content"]
            return name, (msg or "").strip()
        except Exception as exc:
            if attempt == 3:
                print(f"  failed {name}: {str(exc)[:80]}", flush=True)
                return name, ""
            time.sleep(2 ** attempt)


def caption_all():
    caps = ({} if not os.path.exists(CAPTION_PATH)
            else json.load(open(CAPTION_PATH, encoding="utf-8")))
    todo = [n for n in QUESTIONS if not caps.get(n)]
    print(f"{len(todo)} left to caption")
    t0, done = time.time(), 0
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = [pool.submit(caption_one, n) for n in todo]
        for fut in as_completed(futures):
            name, cap = fut.result()
            caps[name] = cap
            done += 1
            if done % 100 == 0:
                rate = done / (time.time() - t0)
                print(f"  {done}/{len(todo)}  {rate:.1f}/s  "
                      f"eta {(len(todo) - done) / rate / 60:.0f} min", flush=True)
                json.dump(caps, open(CAPTION_PATH, "w", encoding="utf-8"),
                          ensure_ascii=False)
    json.dump(caps, open(CAPTION_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    empty = sum(1 for v in caps.values() if not v)
    print(f"{len(caps)} captions, {empty} empty -> {CAPTION_PATH}")
    return caps


if SMOKE:
    # Read each caption against the response it is meant to ground, and against
    # the spans humans marked as hallucinated. If the caption does not contain
    # enough to see that those spans are unsupported, the feature cannot work
    # and there is no point running the other 2,471.
    for row in load_rows("train", "en")[:5]:
        _, cap = caption_one(row["image_name"])
        print("=" * 74)
        print(f"QUESTIONS ASKED OF THIS IMAGE: {QUESTIONS.get(row['image_name'], [])}")
        print(f"\nCAPTION :\n{cap[:900]}")
        print(f"\nRESPONSE: {row['response'][:300]}")
        if row["labels"]:
            print(f"GOLD HALLUCINATIONS: "
                  f"{[row['response'][s['start']:s['end']] for s in row['labels']]}")
    print("\nIf the captions answer the questions specifically and look accurate, "
          "set SMOKE = False and rerun.")
else:
    CAPTIONS = caption_all()

In [ ]:
# ── Cell N1: verify each response sentence against the caption ───────────────
#
# The WiLine RAG framework, with the caption standing in for the retrieved
# chunk: NLI over (premise, hypothesis) pairs, cosine similarity as a fallback
# when NLI returns neutral.
#
# Captions run ~2,400 characters and the part that answers the questions sits at
# the END, so a single truncated NLI pass would throw away the useful half.
# Instead the caption is split into chunks and each response sentence is scored
# against every chunk, taking the best — which is exactly the white paper's
# "a sentence is faithful if at least one chunk entails it".
#
# Runs on whatever captions exist, so it can be validated on a partial run.

import json, os, re, time, gc
import numpy as np
import torch

CAPTION_PATH = f"{OUT_DIR}/captions_gemma4.json"
NLI_DIM = 7
CHUNK_CHARS = 400

CAPTIONS = json.load(open(CAPTION_PATH, encoding="utf-8"))
CAPTIONS = {k: v for k, v in CAPTIONS.items() if v}
print(f"{len(CAPTIONS)} captions available")

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

NLI_ID = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
SIM_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

nli_tok = AutoTokenizer.from_pretrained(NLI_ID)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_ID).to(DEVICE).eval()
sim_model = SentenceTransformer(SIM_ID, device=DEVICE)

LABELS = {nli_model.config.id2label[i].lower(): i
          for i in range(nli_model.config.num_labels)}
I_ENT = LABELS.get("entailment", 0)
I_CON = LABELS.get("contradiction", 2)
print("label map:", LABELS)


# ── sentences and chunks ─────────────────────────────────────────────────────
SENT_END = re.compile(r"[.!?。！？\n]+")

def split_sentences(text):
    """Sentence spans with character offsets, for Latin and CJK punctuation."""
    spans, start = [], 0
    for m in SENT_END.finditer(text):
        end = m.end()
        if text[start:end].strip():
            spans.append((start, end))
        start = end
    if start < len(text) and text[start:].strip():
        spans.append((start, len(text)))
    return spans


def chunk_caption(caption):
    """Paragraph-ish chunks, so the question-answering tail isn't truncated away."""
    parts, buf = [], ""
    for line in caption.split("\n"):
        if len(buf) + len(line) > CHUNK_CHARS and buf.strip():
            parts.append(buf.strip()); buf = line
        else:
            buf += "\n" + line
    if buf.strip():
        parts.append(buf.strip())
    return parts or [caption[:CHUNK_CHARS]]


# Excluded from faithfulness scoring in the white paper: conversational filler,
# markdown artifacts and bare links are not verifiable claims.
FILLER = re.compile(
    r"^\s*(here('s| is)|sure|certainly|of course|i hope|let me know|in summary|"
    r"voici|bien s[uû]r|certamente|ecco|当然|希望|总之)", re.I)

def is_filler(s):
    s = s.strip()
    if len(s.split()) < 5 and len(s) < 30:
        return True
    if FILLER.match(s) or re.fullmatch(r"[\s*#\-_=|`~\[\]()]*", s):
        return True
    return s.lower().startswith(("http", "source:", "!["))


@torch.no_grad()
def nli_probs(premises, hypotheses, batch_size=64):
    out = []
    for i in range(0, len(premises), batch_size):
        enc = nli_tok(premises[i:i + batch_size], hypotheses[i:i + batch_size],
                      return_tensors="pt", truncation=True, max_length=256,
                      padding=True).to(DEVICE)
        out.append(torch.softmax(nli_model(**enc).logits, dim=-1).cpu().numpy())
    return np.concatenate(out, axis=0) if out else np.zeros((0, 3))


# ── extraction ───────────────────────────────────────────────────────────────
def extract_nli(split, lang, limit=None):
    rows = load_rows(split, lang)
    if limit:
        rows = rows[:limit]
    rows = [r for r in rows if r["image_name"] in CAPTIONS]
    print(f"{split}/{lang}: {len(rows)} rows with captions")
    store, t0 = {}, time.time()
    for i, row in enumerate(rows):
        resp = row["response"]
        chunks = chunk_caption(CAPTIONS[row["image_name"]])
        spans = split_sentences(resp)
        if not spans:
            continue
        sents = [resp[a:b].strip() for a, b in spans]
        # every sentence against every chunk
        prem = [c for _ in sents for c in chunks]
        hyp = [s for s in sents for _ in chunks]
        p = nli_probs(prem, hyp).reshape(len(sents), len(chunks), -1)
        emb = sim_model.encode(chunks + sents, convert_to_numpy=True,
                               normalize_embeddings=True, show_progress_bar=False)
        cos = emb[len(chunks):] @ emb[:len(chunks)].T
        feats = np.zeros((len(spans), NLI_DIM), dtype=np.float32)
        for j, s in enumerate(sents):
            feats[j, 0] = p[j, :, I_ENT].max()     # best entailment over chunks
            feats[j, 1] = p[j, :, I_ENT].mean()
            feats[j, 2] = p[j, :, I_CON].max()     # any chunk contradicting it
            feats[j, 3] = cos[j].max()
            feats[j, 4] = float(is_filler(s))
            feats[j, 5] = j / max(len(sents) - 1, 1)
            feats[j, 6] = min(len(s) / 200.0, 1.0)
        store[row["id"] + "|off"] = np.array(spans, dtype=np.int32)
        store[row["id"] + "|f"] = feats
        if (i + 1) % 500 == 0:
            rate = (i + 1) / (time.time() - t0)
            print(f"  {i + 1}/{len(rows)} {rate:.1f}/s "
                  f"eta {(len(rows) - i - 1) / rate / 60:.0f} min", flush=True)
    path = f"{OUT_DIR}/nli_feats_{split}_{lang}.npz"
    np.savez_compressed(path, **store)
    print(f"{split}/{lang}: {len(store) // 2} saved, "
          f"{(time.time() - t0) / 60:.1f} min -> {path}", flush=True)


extract_nli("train", "en")
gc.collect(); torch.cuda.empty_cache()


# ── Cell N2: does the entailment signal exist? ───────────────────────────────
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

d = np.load(f"{OUT_DIR}/nli_feats_train_en.npz")
gold_rows = {json.loads(l)["id"]: json.loads(l)
             for l in open(f"{DISTRIB}/shroom-vision.train.en.labeled.jsonl")}

ys, xs, groups = [], [], []
for key in [k for k in d.files if k.endswith("|f")]:
    rid = key[:-2]
    row, offs, feats = gold_rows[rid], d[rid + "|off"], d[key]
    gold = np.zeros(len(row["response"]))
    for s in row["labels"]:
        seg = gold[s["start"]:s["end"]]
        gold[s["start"]:s["end"]] = np.maximum(seg, s["prob"])
    for (a, b), f in zip(offs, feats):
        ys.append(gold[a:b].max() > 0 if b > a else False)
        xs.append(f); groups.append(rid)

ys, xs, groups = np.array(ys), np.array(xs), np.array(groups)
names = ["entail_max", "entail_mean", "contradict_max", "cosine_max",
         "filler", "position", "length"]
print(f"\n{ys.sum()} hallucinated / {len(ys)} sentences "
      f"({100 * ys.mean():.1f}%), {len(set(groups))} responses\n")
print(f"{'feature':<16}{'AUC':>8}")
for i, n in enumerate(names):
    a = roc_auc_score(ys, xs[:, i])
    print(f"{n:<16}{max(a, 1 - a):>8.3f}")

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
              .split(xs, ys, groups))
mu, sd = xs[tr].mean(0), xs[tr].std(0) + 1e-2
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit((xs[tr] - mu) / sd, ys[tr])
print(f"\nlogistic probe, grouped by response: "
      f"AUC {roc_auc_score(ys[te], clf.predict_proba((xs[te] - mu) / sd)[:, 1]):.3f}")
print("this is SENTENCE-level, so not directly comparable to the token-level "
      "0.636 from the visual features")

In [ ]:
# ── Cell K1: batched captioning with Qwen2.5-VL-7B on the Kaggle GPU ─────────
#
# Same question-conditioned prompt as the gemma4 route, so the two captioners
# stay comparable. The only reason this is viable at all is batching: one image
# at a time would take ~24 h, eight at a time gets it to ~3 h.
#
# Writes to a SEPARATE file from the gemma4 captions, so if Eduardo fixes the
# Ollama parallelism later you end up with both and can compare.

SMOKE = False          # 8 images and stop
BATCH_IMAGES = 16
MAX_NEW = 400         # captions ran ~600 tokens on gemma4; 400 is enough for
                      # the description plus the question answers
MAX_QUESTIONS = 6

import json, os, time, gc
from collections import defaultdict
import numpy as np
import torch
from PIL import Image

CAPTION_PATH = f"{OUT_DIR}/captions_qwen7b.json"
VLM_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

import gc, torch
for _v in ["vlm", "model", "nli_model", "sim_model"]:
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

# 4-bit keeps a 7B inside the T4's 15 GB with room for batched activations.
import subprocess
subprocess.run("pip install -q -U bitsandbytes", shell=True)
from transformers import BitsAndBytesConfig, AutoProcessor
try:
    from transformers import Qwen2_5_VLForConditionalGeneration as VLMClass
except ImportError:  # older transformers
    from transformers import Qwen2VLForConditionalGeneration as VLMClass
    VLM_ID = "Qwen/Qwen2-VL-7B-Instruct"
    print("falling back to Qwen2-VL-7B")

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                           bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
vlm = VLMClass.from_pretrained(VLM_ID, device_map="auto", quantization_config=quant)
vlm.eval()
vproc = AutoProcessor.from_pretrained(VLM_ID, min_pixels=128 * 28 * 28,
                                      max_pixels=512 * 28 * 28)
# left padding is required for batched generation, or short sequences get their
# generated tokens attached after the pad run instead of after the prompt
vproc.tokenizer.padding_side = "left"
print(f"{VLM_ID} loaded")


def load_rows(split, lang):
    kind = "labeled" if split == "train" else "unlabeled"
    with open(f"{DISTRIB}/shroom-vision.{split}.{lang}.{kind}.jsonl",
              encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


def questions_by_image():
    out = defaultdict(list)
    for split in ["train", "test"]:
        for lang in LANGS:
            for row in load_rows(split, lang):
                q = row["prompt"].strip()
                if q not in out[row["image_name"]]:
                    out[row["image_name"]].append(q)
    return dict(out)


QUESTIONS = questions_by_image()
print(f"{len(QUESTIONS)} unique images")


def build_prompt(questions):
    listed = "\n".join(f"{i}. {q}" for i, q in enumerate(questions[:MAX_QUESTIONS], 1))
    return (
        "Describe this image factually and in detail. State the objects present, "
        "their colours, shapes and other visible attributes, how many of each "
        "there are, where they are relative to each other, and transcribe any "
        "text visible in the image.\n\n"
        "Your description must explicitly answer each of these questions:\n"
        f"{listed}\n\n"
        "Describe only what you can actually see. If something cannot be "
        "determined from the image, say so plainly rather than guessing."
    )


@torch.no_grad()
def caption_batch(names):
    images, texts, ok = [], [], []
    for name in names:
        try:
            img = Image.open(os.path.join(IMAGE_DIR, name)).convert("RGB")
        except Exception as exc:
            print(f"  skip {name}: {exc}")
            continue
        messages = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": build_prompt(QUESTIONS.get(name, []))}]}]
        texts.append(vproc.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True))
        images.append(img)
        ok.append(name)
    if not ok:
        return {}
    inputs = vproc(text=texts, images=images, return_tensors="pt",
                   padding=True).to(vlm.device)
    out = vlm.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[1]:]
    decoded = vproc.tokenizer.batch_decode(gen, skip_special_tokens=True)
    return {n: t.strip() for n, t in zip(ok, decoded)}


def caption_all():
    caps = ({} if not os.path.exists(CAPTION_PATH)
            else json.load(open(CAPTION_PATH, encoding="utf-8")))
    todo = [n for n in QUESTIONS if not caps.get(n)]
    print(f"{len(todo)} left to caption, batch {BATCH_IMAGES}", flush=True)
    t0 = time.time()
    for i in range(0, len(todo), BATCH_IMAGES):
        caps.update(caption_batch(todo[i:i + BATCH_IMAGES]))
        done = min(i + BATCH_IMAGES, len(todo))
        # Save after EVERY batch, not every tenth. The file is small and a
        # dropped session should cost one batch, not ten.
        tmp = CAPTION_PATH + ".tmp"
        with open(tmp, "w", encoding="utf-8") as fh:
            json.dump(caps, fh, ensure_ascii=False)
        os.replace(tmp, CAPTION_PATH)      # atomic: never a half-written file
        rate = done / (time.time() - t0)
        print(f"  {done}/{len(todo)}  {rate:.2f}/s  "
              f"eta {(len(todo) - done) / rate / 60:.0f} min", flush=True)
    print(f"{len(caps)} captions -> {CAPTION_PATH}")
    return caps


if SMOKE:
    # One batch, timed. Two things must hold: the captions must be distinct
    # (a padding bug shows up as identical or truncated outputs), and they must
    # answer the questions.
    names = [r["image_name"] for r in load_rows("train", "en")[:BATCH_IMAGES]]
    t0 = time.time()
    got = caption_batch(names)
    dt = time.time() - t0
    print(f"\n{len(got)} captions in {dt:.0f}s "
          f"-> {dt / max(len(got), 1):.1f}s each, "
          f"eta for 2476: {2476 * dt / max(len(got), 1) / 60:.0f} min\n")
    for name, cap in list(got.items())[:2]:
        print("=" * 74)
        print(f"QUESTIONS: {QUESTIONS.get(name, [])[:MAX_QUESTIONS]}")
        print(f"CAPTION ({len(cap)} chars):\n{cap[:800]}\n")
    lens = [len(c) for c in got.values()]
    print(f"caption lengths: {lens}")
    print("all distinct:", len(set(got.values())) == len(got))
else:
    CAPTIONS = caption_all()

In [ ]:
import json, os
for f in sorted(os.listdir("/kaggle/working")):
    if f.endswith((".json", ".npz", ".pt")):
        print(f"{os.path.getsize('/kaggle/working/' + f)/1e6:8.1f} MB  {f}")
print()
for p in ["captions_qwen7b.json", "captions_gemma4.json"]:
    path = f"/kaggle/working/{p}"
    if os.path.exists(path):
        c = json.load(open(path, encoding="utf-8"))
        print(f"{p}: {len(c)} entries, {sum(1 for v in c.values() if v)} non-empty")

In [ ]:
# ── Cell N1: verify each response sentence against the caption ───────────────
#
# The WiLine RAG framework with the caption standing in for the retrieved
# chunk: NLI over (premise, hypothesis) pairs, cosine similarity alongside.
#
# Captions run ~1,500 characters and the part answering the questions sits at
# the END, so one truncated NLI pass would throw away the useful half. The
# caption is chunked and every response sentence is scored against every chunk,
# taking the best — the white paper's "faithful if at least one chunk entails it".

import json, os, re, time, gc
import numpy as np
import torch

CAPTION_PATH = f"{OUT_DIR}/captions_qwen7b.json"
NLI_DIM = 7
CHUNK_CHARS = 400

CAPTIONS = {k: v for k, v in
            json.load(open(CAPTION_PATH, encoding="utf-8")).items() if v}
print(f"{len(CAPTIONS)} captions available")

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

NLI_ID = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
SIM_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
nli_tok = AutoTokenizer.from_pretrained(NLI_ID)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_ID).to(DEVICE).eval()
sim_model = SentenceTransformer(SIM_ID, device=DEVICE)
LABELS = {nli_model.config.id2label[i].lower(): i
          for i in range(nli_model.config.num_labels)}
I_ENT, I_CON = LABELS.get("entailment", 0), LABELS.get("contradiction", 2)
print("label map:", LABELS)


def load_rows(split, lang):
    kind = "labeled" if split == "train" else "unlabeled"
    with open(f"{DISTRIB}/shroom-vision.{split}.{lang}.{kind}.jsonl",
              encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


SENT_END = re.compile(r"[.!?。！？\n]+")

def split_sentences(text):
    spans, start = [], 0
    for m in SENT_END.finditer(text):
        end = m.end()
        if text[start:end].strip():
            spans.append((start, end))
        start = end
    if start < len(text) and text[start:].strip():
        spans.append((start, len(text)))
    return spans


def chunk_caption(caption):
    parts, buf = [], ""
    for line in caption.split("\n"):
        if len(buf) + len(line) > CHUNK_CHARS and buf.strip():
            parts.append(buf.strip()); buf = line
        else:
            buf += "\n" + line
    if buf.strip():
        parts.append(buf.strip())
    return parts or [caption[:CHUNK_CHARS]]


FILLER = re.compile(
    r"^\s*(here('s| is)|sure|certainly|of course|i hope|let me know|in summary|"
    r"voici|bien s[uû]r|certamente|ecco|当然|希望|总之)", re.I)

def is_filler(s):
    s = s.strip()
    if len(s.split()) < 5 and len(s) < 30:
        return True
    if FILLER.match(s) or re.fullmatch(r"[\s*#\-_=|`~\[\]()]*", s):
        return True
    return s.lower().startswith(("http", "source:", "!["))


@torch.no_grad()
def nli_probs(premises, hypotheses, batch_size=64):
    out = []
    for i in range(0, len(premises), batch_size):
        enc = nli_tok(premises[i:i + batch_size], hypotheses[i:i + batch_size],
                      return_tensors="pt", truncation=True, max_length=256,
                      padding=True).to(DEVICE)
        out.append(torch.softmax(nli_model(**enc).logits, dim=-1).cpu().numpy())
    return np.concatenate(out, axis=0) if out else np.zeros((0, 3))


def extract_nli(split, lang):
    rows = [r for r in load_rows(split, lang) if r["image_name"] in CAPTIONS]
    store, t0 = {}, time.time()
    for i, row in enumerate(rows):
        resp = row["response"]
        chunks = chunk_caption(CAPTIONS[row["image_name"]])
        spans = split_sentences(resp)
        if not spans:
            continue
        sents = [resp[a:b].strip() for a, b in spans]
        prem = [c for _ in sents for c in chunks]
        hyp = [s for s in sents for _ in chunks]
        p = nli_probs(prem, hyp).reshape(len(sents), len(chunks), -1)
        emb = sim_model.encode(chunks + sents, convert_to_numpy=True,
                               normalize_embeddings=True, show_progress_bar=False)
        cos = emb[len(chunks):] @ emb[:len(chunks)].T
        feats = np.zeros((len(spans), NLI_DIM), dtype=np.float32)
        for j, s in enumerate(sents):
            feats[j, 0] = p[j, :, I_ENT].max()
            feats[j, 1] = p[j, :, I_ENT].mean()
            feats[j, 2] = p[j, :, I_CON].max()
            feats[j, 3] = cos[j].max()
            feats[j, 4] = float(is_filler(s))
            feats[j, 5] = j / max(len(sents) - 1, 1)
            feats[j, 6] = min(len(s) / 200.0, 1.0)
        store[row["id"] + "|off"] = np.array(spans, dtype=np.int32)
        store[row["id"] + "|f"] = feats
        if (i + 1) % 500 == 0:
            rate = (i + 1) / (time.time() - t0)
            print(f"  {i + 1}/{len(rows)} {rate:.1f}/s "
                  f"eta {(len(rows) - i - 1) / rate / 60:.0f} min", flush=True)
    path = f"{OUT_DIR}/nli_feats_{split}_{lang}.npz"
    np.savez_compressed(path, **store)
    print(f"{split}/{lang}: {len(store) // 2} saved, "
          f"{(time.time() - t0) / 60:.1f} min -> {path}", flush=True)


for split in ["test", "train"]:
    for lang in LANGS:
        extract_nli(split, lang)
        gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ── FINAL TAGGER: XLM-R + visual ablation + caption entailment ──────────────
#
# Two flags give the four ablation rows. Split, seed, hyperparameters and code
# path are identical across them, so only the features can explain a difference.
#
#   USE_VIS=False USE_NLI=False   text only        (matched control)
#   USE_VIS=True  USE_NLI=False   + visual ablation
#   USE_VIS=False USE_NLI=True    + caption entailment
#   USE_VIS=True  USE_NLI=True    + both
#
# Both feature sets route through CHARACTERS: the visual features are indexed by
# Qwen tokens, the NLI features by sentences, and neither lines up with XLM-R's
# tokenization. Characters are the common ground and what the scorer measures.

USE_VIS = True
USE_NLI = True
VIS_DIM, NLI_DIM = 38, 7
FEAT_PROJ = 64
TAG = f"vis{int(USE_VIS)}nli{int(USE_NLI)}"

import json, os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, get_linear_schedule_with_warmup
import random

print(f"USE_VIS={USE_VIS}  USE_NLI={USE_NLI}  tag={TAG}")

def load(lang, split_name):
    kind = "labeled" if split_name == "train" else "unlabeled"
    with open(f"{DISTRIB}/shroom-vision.{split_name}.{lang}.{kind}.jsonl",
              encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


def split_dev(rows, fraction=0.2, seed=SEED):
    """Stratified split preserving the clean / hallucinated ratio."""
    clean = [r for r in rows if not r["labels"]]
    dirty = [r for r in rows if r["labels"]]
    rng = random.Random(seed)
    rng.shuffle(clean); rng.shuffle(dirty)
    nc, nd = int(len(clean) * fraction), int(len(dirty) * fraction)
    dev = clean[:nc] + dirty[:nd]
    train = clean[nc:] + dirty[nd:]
    rng.shuffle(dev); rng.shuffle(train)
    return train, dev

def load_char_features(prefix, dim):
    """id -> (n_chars, dim). Sentence or token spans expanded onto characters."""
    out = {}
    for split in ["train", "test"]:
        for lang in LANGS:
            path = f"{OUT_DIR}/{prefix}_{split}_{lang}.npz"
            if not os.path.exists(path):
                print(f"  missing {path}"); continue
            d = np.load(path)
            for key in d.files:
                if not key.endswith("|f"):
                    continue
                rid = key[:-2]
                offs, feats = d[rid + "|off"], d[key]
                if len(offs) == 0:
                    continue
                arr = np.zeros((int(offs[-1][1]), dim), dtype=np.float32)
                for (a, b), f in zip(offs, feats):
                    arr[a:b] = f
                out[rid] = arr
    return out


VIS = load_char_features("hidden_feats", VIS_DIM) if USE_VIS else {}
NLI = load_char_features("nli_feats", NLI_DIM) if USE_NLI else {}
print(f"visual: {len(VIS)}   nli: {len(NLI)}")

FEAT_DIM = (VIS_DIM if USE_VIS else 0) + (NLI_DIM if USE_NLI else 0)


def _stats(store, dim):
    if not store:
        return np.zeros(dim, np.float32), np.ones(dim, np.float32)
    sample = np.concatenate([v for _, v in list(store.items())[:2000]], axis=0)
    # Floor the deviation: near-constant columns would otherwise be divided by
    # ~zero and blow up into NaN.
    return sample.mean(0), np.maximum(sample.std(0), 1e-2)


VIS_MU, VIS_SD = _stats(VIS, VIS_DIM)
NLI_MU, NLI_SD = _stats(NLI, NLI_DIM)


def features_for_tokens(rid, offsets):
    parts, present = [], 1.0
    for store, dim, mu, sd, on in [(VIS, VIS_DIM, VIS_MU, VIS_SD, USE_VIS),
                                   (NLI, NLI_DIM, NLI_MU, NLI_SD, USE_NLI)]:
        if not on:
            continue
        arr = store.get(rid)
        block = np.zeros((len(offsets), dim), dtype=np.float32)
        if arr is None:
            present = 0.0
        else:
            for i, (a, b) in enumerate(offsets):
                b = min(b, len(arr))
                if b > a:
                    block[i] = arr[a:b].mean(axis=0)
            block = (block - mu) / sd
        parts.append(block)
    if not parts:
        return np.zeros((len(offsets), 0), dtype=np.float32), 1.0
    return np.concatenate(parts, axis=1), present


class FinalData(Dataset):
    def __init__(self, rows, labeled=True):
        self.rows, self.labeled = rows, labeled
        self.cache = [encode(r) for r in rows]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        row = self.rows[i]
        enc, keep = self.cache[i]
        offs = [enc["offset_mapping"][k] for k in keep]
        feat, present = features_for_tokens(row["id"], offs)
        item = {
            "input_ids": torch.tensor(enc["input_ids"]),
            "attention_mask": torch.tensor(enc["attention_mask"]),
            "keep": torch.tensor(keep, dtype=torch.long),
            "feat": torch.tensor(feat),
            "present": torch.tensor(present, dtype=torch.float),
        }
        if self.labeled:
            prob, cat = char_targets(row)
            item["tok_prob"] = torch.tensor(
                [float(prob[a:b].max()) if b > a else 0.0 for a, b in offs],
                dtype=torch.float)
            tc = []
            for a, b in offs:
                seg = cat[a:b]; seg = seg[seg >= 0]
                tc.append(int(np.bincount(seg).argmax()) if len(seg) else -100)
            item["tok_cat"] = torch.tensor(tc, dtype=torch.long)
        return item


def collate_final(batch):
    pad = tokenizer.pad_token_id
    n = len(batch)
    maxlen = max(len(b["input_ids"]) for b in batch)
    maxkeep = max(len(b["keep"]) for b in batch)
    out = {
        "input_ids": torch.full((n, maxlen), pad, dtype=torch.long),
        "attention_mask": torch.zeros((n, maxlen), dtype=torch.long),
        "keep": torch.zeros((n, maxkeep), dtype=torch.long),
        "keep_mask": torch.zeros((n, maxkeep), dtype=torch.bool),
        "feat": torch.zeros((n, maxkeep, FEAT_DIM)),
        "present": torch.stack([b["present"] for b in batch]),
    }
    has_labels = "tok_prob" in batch[0]
    if has_labels:
        out["tok_prob"] = torch.zeros((n, maxkeep))
        out["tok_cat"] = torch.full((n, maxkeep), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L, K = len(b["input_ids"]), len(b["keep"])
        out["input_ids"][i, :L] = b["input_ids"]
        out["attention_mask"][i, :L] = b["attention_mask"]
        out["keep"][i, :K] = b["keep"]
        out["keep_mask"][i, :K] = True
        if FEAT_DIM:
            out["feat"][i, :K] = b["feat"]
        if has_labels:
            out["tok_prob"][i, :K] = b["tok_prob"]
            out["tok_cat"][i, :K] = b["tok_cat"]
    return out


class FinalTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_ID)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        if FEAT_DIM:
            self.feat_proj = nn.Sequential(
                nn.Linear(FEAT_DIM, FEAT_PROJ), nn.GELU(), nn.LayerNorm(FEAT_PROJ))
            h += FEAT_PROJ
        self.span_head = nn.Linear(h, 1)
        self.cat_head = nn.Linear(h, len(CATEGORIES))
    def forward(self, input_ids, attention_mask, keep, keep_mask, feat, present):
        hidden = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state
        idx = keep.unsqueeze(-1).expand(-1, -1, hidden.size(-1))
        tok = torch.gather(hidden, 1, idx)
        if FEAT_DIM:
            tok = torch.cat([tok, self.feat_proj(feat) * present.view(-1, 1, 1)], dim=-1)
        tok = self.dropout(tok)
        return self.span_head(tok).squeeze(-1), self.cat_head(tok)


def run_epoch_final(model, loader, optimizer=None, scheduler=None, log_every=50):
    train = optimizer is not None
    model.train() if train else model.eval()
    bce = nn.BCEWithLogitsLoss(reduction="none")
    ce = nn.CrossEntropyLoss(ignore_index=-100)
    total, nb = 0.0, 0
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.set_grad_enabled(train):
            span_logit, cat_logit = model(
                batch["input_ids"], batch["attention_mask"], batch["keep"],
                batch["keep_mask"], batch["feat"], batch["present"])
            m = batch["keep_mask"]
            l_span = (bce(span_logit, batch["tok_prob"]) * m).sum() / m.sum().clamp(min=1)
            valid = (batch["tok_cat"] != -100).any()
            l_cat = (ce(cat_logit.reshape(-1, len(CATEGORIES)),
                        batch["tok_cat"].reshape(-1)) if valid
                     else span_logit.sum() * 0.0)
            loss = l_span + 0.5 * l_cat
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total += loss.item(); nb += 1
        if train and step % log_every == 0:
            print(f"  step {step}/{len(loader)}  loss {total / nb:.4f}", flush=True)
    return total / max(nb, 1)


@torch.no_grad()
def predict_final(model, rows):
    model.eval()
    ds = FinalData(rows, labeled=False)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False, collate_fn=collate_final)
    results, cursor = [], 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        span_logit, cat_logit = model(
            batch["input_ids"], batch["attention_mask"], batch["keep"],
            batch["keep_mask"], batch["feat"], batch["present"])
        probs = torch.sigmoid(span_logit).cpu().numpy()
        cats = cat_logit.argmax(-1).cpu().numpy()
        for i in range(len(probs)):
            row = rows[cursor]
            enc, keep = ds.cache[cursor]
            cursor += 1
            offs = [enc["offset_mapping"][k] for k in keep]
            n = len(row["response"])
            cp = np.zeros(n, dtype=np.float32)
            cc = np.zeros(n, dtype=np.int64)
            for j, (a, b) in enumerate(offs):
                cp[a:min(b, n)] = probs[i][j]
                cc[a:min(b, n)] = cats[i][j]
            results.append((cp, cc))
    return results


# ── train ───────────────────────────────────────────────────────────────────
train_rows, dev_rows = [], []
for lang in LANGS:
    tr, dv = split_dev(load(lang, "train"))
    train_rows += tr; dev_rows += dv
print(f"train={len(train_rows)}  dev={len(dev_rows)}  feat_dim={FEAT_DIM}")

model = FinalTagger().to(DEVICE)
if MODEL_ID.endswith("large"):
    model.encoder.embeddings.requires_grad_(False)

loader = DataLoader(FinalData(train_rows), batch_size=BATCH, shuffle=True,
                    collate_fn=collate_final)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=LR, weight_decay=0.01, foreach=False)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(0.1 * len(loader) * EPOCHS), len(loader) * EPOCHS)

for epoch in range(EPOCHS):
    loss = run_epoch_final(model, loader, optimizer, scheduler)
    print(f"epoch {epoch + 1}: loss {loss:.4f}", flush=True)
    torch.save(model.state_dict(), f"{OUT_DIR}/tagger_{TAG}.pt")

print(f"\n{'lang':<6}{'thr':>6}{'Cor':>9}{'Cor_lbl':>9}{'IoU':>9}   (floor)")
thresholds, summary = {}, {}
for lang in LANGS:
    rows = [r for r in dev_rows if r["language"] == lang]
    cps = predict_final(model, rows)
    thr, cor, scores = tune_threshold(rows, cps)
    thresholds[lang] = thr; summary[lang] = scores
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in rows]
    floor = evaluate(refs, [{"id": r["id"], "labels": []} for r in rows])["Cor"]
    print(f"{lang:<6}{thr:>6.2f}{scores['Cor']:>9.3f}{scores['Cor_lbl']:>9.3f}"
          f"{scores['IoU']:>9.3f}   {floor:.3f}")

json.dump({"tag": TAG, "use_vis": USE_VIS, "use_nli": USE_NLI,
           "thresholds": thresholds,
           "dev": {l: {k: float(v) for k, v in s.items()} for l, s in summary.items()}},
          open(f"{OUT_DIR}/result_{TAG}.json", "w"), indent=1)

for lang in LANGS:
    rows = load(lang, "test")
    cps = predict_final(model, rows)
    preds = build_preds(rows, cps, thresholds[lang])
    path = f"{OUT_DIR}/predictions_{TAG}_{lang}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for p in preds:
            fh.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"{lang}: {len(preds)} rows, "
          f"{sum(len(p['labels']) for p in preds)} spans -> {path}")

In [ ]:
from IPython.display import FileLink
!cd /kaggle/working && zip -q preds_v4.zip predictions_vis1nli1_*.jsonl && unzip -l preds_v4.zip
FileLink('preds_v4.zip')

In [ ]:
from IPython.display import FileLink
!cd /kaggle/working && zip -q feats.zip hidden_feats_*.npz nli_feats_*.npz && ls -la feats.zip
FileLink('feats.zip')

In [ ]:
def load(lang, split_name):
    kind = "labeled" if split_name == "train" else "unlabeled"
    with open(f"{DISTRIB}/shroom-vision.{split_name}.{lang}.{kind}.jsonl",
              encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]

CATEGORIES = ["invention", "mischaracterization", "OCR", "miscounting", "other"]

In [2]:
# ── Judge probe: can gemma4 mark the spans directly? ────────────────────────
#
# The approach that won the text-only edition: show a model the evidence and
# the response, ask which parts are unsupported, map back to character offsets.
# Our first attempt failed with a prompt demanding JSON and a parser that
# discarded anything malformed — so it was never really tested.
#
# 100 labelled training examples, scored with the official metric, so the
# number is directly comparable to the tagger.

import base64, io, json, os, re, time, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
from PIL import Image

N, WORKERS = 100, 4
BASE = "https://inference.wiline.com/v1/chat/completions"
MODEL = "gemma4"
IMAGE_DIR = "/kaggle/working/shroom-vis-images"
from kaggle_secrets import UserSecretsClient
KEY = UserSecretsClient().get_secret("WEC_API_KEY")

# Quoting substrings is far more robust than asking for character offsets,
# which models cannot count reliably. We match the quotes back ourselves.
PROMPT = """You are checking an AI-generated answer against the image it describes.

QUESTION: {question}

ANSWER: {response}

Some parts of the answer may not be supported by the image. List ONLY those parts.

Rules:
- Quote each unsupported part EXACTLY as it appears in the answer, character for character.
- Quote the shortest span that carries the error, not the whole sentence.
- Give each one a category: invention (something not in the image),
  mischaracterization (something described wrongly), OCR (text misread),
  miscounting (wrong quantity), other.
- If everything in the answer is supported by the image, output NONE.

Format, one per line, nothing else:
<exact quote> ||| <category>"""


def encode_image(name, max_side=1024):
    img = Image.open(os.path.join(IMAGE_DIR, name)).convert("RGB")
    img.thumbnail((max_side, max_side))
    buf = io.BytesIO(); img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()


def ask(row):
    body = json.dumps({
        "model": MODEL, "max_tokens": 400, "think": False,
        "num_ctx": 8192, "temperature": 0,
        "messages": [{"role": "user", "content": [
            {"type": "text", "text": PROMPT.format(question=row["prompt"],
                                                   response=row["response"])},
            {"type": "image_url", "image_url": {
                "url": f"data:image/jpeg;base64,{encode_image(row['image_name'])}"}}]}],
    }).encode()
    for attempt in range(3):
        try:
            req = urllib.request.Request(BASE, data=body,
                headers={"Authorization": f"Bearer {KEY}",
                         "Content-Type": "application/json"})
            with urllib.request.urlopen(req, timeout=180) as r:
                return row["id"], (json.load(r)["choices"][0]["message"]["content"] or "")
        except Exception:
            if attempt == 2: return row["id"], ""
            time.sleep(2 ** attempt)


def parse(raw, response):
    """Map quoted substrings back to offsets; models alter spacing when quoting."""
    spans, used = [], []
    for line in raw.splitlines():
        line = line.strip().lstrip("-*0123456789. ")
        if not line or line.upper().startswith("NONE"):
            continue
        quote, _, cat = line.partition("|||")
        quote = quote.strip().strip('"“”')
        cat = next((c for c in CATEGORIES if c.lower() in cat.strip().lower()),
                   "mischaracterization")
        if len(quote) < 2:
            continue
        start = response.find(quote)
        if start < 0:
            flat = re.sub(r"\s+", " ", quote)
            m = re.search(re.escape(flat).replace(r"\ ", r"\s+"), response)
            if not m: continue
            start, end = m.start(), m.end()
        else:
            end = start + len(quote)
        if any(start < e and s < end for s, e in used):
            continue
        used.append((start, end))
        spans.append({"start": start, "end": end, "prob": 1.0, "label": cat})
    return sorted(spans, key=lambda s: s["start"])


rows = load("en", "train")[:N]
print(f"{len(rows)} examples, model={MODEL}, {WORKERS} workers")
raws, t0 = {}, time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for i, fut in enumerate(as_completed([pool.submit(ask, r) for r in rows]), 1):
        rid, raw = fut.result(); raws[rid] = raw
        if i % 20 == 0:
            rate = i / (time.time() - t0)
            print(f"  {i}/{len(rows)}  {rate:.2f}/s  eta {(len(rows)-i)/rate/60:.0f} min",
                  flush=True)
print(f"done in {(time.time()-t0)/60:.1f} min")

refs, preds, empty, unmatched = [], [], 0, 0
for row in rows:
    raw = raws.get(row["id"], "")
    if not raw.strip(): empty += 1
    spans = parse(raw, row["response"])
    if raw.strip() and not spans and "NONE" not in raw.upper(): unmatched += 1
    refs.append({"id": row["id"], "labels": row["labels"],
                 "text_len": len(row["response"])})
    preds.append({"id": row["id"], "labels": spans})

s = evaluate(refs, preds)
floor = evaluate(refs, [{"id": r["id"], "labels": []} for r in refs])
print(f"\nCor       {s['Cor']:.3f}")
print(f"Cor_lbl   {s['Cor_lbl']:.3f}")
print(f"IoU       {s['IoU']:.3f}")
print(f"mark_none {floor['Cor']:.3f}   (floor)")
print(f"\nsupervised tagger: 0.34 en test, 0.42 it test")
print(f"empty replies {empty}/{len(rows)}   quotes matching nothing {unmatched}")

for row in rows[:3]:
    print("=" * 74)
    print("RAW :", raws.get(row["id"], "")[:250].replace("\n", " | "))
    print("PRED:", [row["response"][x["start"]:x["end"]]
                    for x# =============================================================================
#  Verifica o AUROC 0.485 do output-level likelihood (§5.1 do paper).
#
#  Cole isto numa celula do Kaggle com GPU ligada e rode. Ele faz tudo:
#  baixa dados+imagens, roda os dois forward passes do Qwen2-VL-2B, treina a
#  sonda e IMPRIME O NUMERO FINAL. Nao precisa voltar para o Claude.
#
#  Tempo: SMOKE=True leva ~2 min. SMOKE=False leva 2-4 h (ingles, treino).
#  Ao terminar, baixe visual_feats_train_en.npz para nunca perder de novo.
# =============================================================================
SMOKE = True          # rode assim primeiro; se o alinhamento passar, ponha False
LANG  = "en"          # o paper reporta ingles
LIMIT = None          # cap de exemplos, ou None

import os, sys, json, time, gc, subprocess
import numpy as np

# ── 1. dados + imagens (mesmo zip) ───────────────────────────────────────────
if not os.path.isdir("/kaggle/working/distrib"):
    subprocess.run("wget -q https://a3s.fi/mickusti-2007780-pub/shroom-visions-data.zip"
                   " -O /tmp/data.zip && unzip -oq /tmp/data.zip -d /kaggle/working/",
                   shell=True, check=True)
DISTRIB   = "/kaggle/working/distrib"
IMAGE_DIR = "/kaggle/working/shroom-vis-images"
print("jsonl:", len(os.listdir(DISTRIB)), "| imagens:", len(os.listdir(IMAGE_DIR)))

# ── 2. modelo (identico ao day4_extract_visual.py) ───────────────────────────
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
MODEL_ID   = "Qwen/Qwen2-VL-2B-Instruct"
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4"))
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
tokenizer = processor.tokenizer
print("modelo carregado em", DEVICE)

# ── 3. scoring: dois passes, com e sem imagem ────────────────────────────────
@torch.no_grad()
def score_response(image, prompt, response):
    enc = tokenizer(response, return_offsets_mapping=True, add_special_tokens=False)
    resp_ids, offsets = enc["input_ids"], enc["offset_mapping"]
    if not resp_ids:
        return None, None

    def run(with_image):
        content = ([{"type": "image"}] if with_image else []) + \
                  [{"type": "text", "text": prompt}]
        prefix = processor.apply_chat_template(
            [{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
        kw  = {"text": [prefix + response], "return_tensors": "pt"}
        pkw = {"text": [prefix], "return_tensors": "pt"}
        if with_image:
            kw["images"] = [image]; pkw["images"] = [image]
        inputs   = processor(**kw).to(DEVICE)
        n_prefix = processor(**pkw)["input_ids"].shape[1]
        ids = inputs["input_ids"][0]
        if len(ids) - n_prefix != len(resp_ids):     # alinhamento quebrou
            return None
        step     = model(**inputs).logits[0].float()[n_prefix - 1:-1]
        logprobs = torch.log_softmax(step, dim=-1)
        target   = ids[n_prefix:]
        return (logprobs.gather(1, target.unsqueeze(1)).squeeze(1).cpu().numpy(),
                (-(logprobs.exp() * logprobs).sum(-1)).cpu().numpy())

    a, b = run(True), run(False)
    if a is None or b is None:
        return None, None
    (li, ei), (ln, en) = a, b
    feats = np.stack([li, ei, ln, en, li - ln, ei - en], axis=1)
    return np.array(offsets, dtype=np.int32), feats.astype(np.float32)

rows = [json.loads(l) for l in open(f"{DISTRIB}/shroom-vision.train.{LANG}.labeled.jsonl")]
if LIMIT: rows = rows[:LIMIT]

if SMOKE:
    for row in rows[:5]:
        img = Image.open(os.path.join(IMAGE_DIR, row["image_name"])).convert("RGB")
        off, f = score_response(img, row["prompt"], row["response"])
        if f is None:
            print(row["id"], "ALINHAMENTO FALHOU"); continue
        d = f[:, 4]
        print(f'{row["id"]}: {len(f)} tokens, d_logp mean {d.mean():+.3f} '
              f'min {d.min():+.3f} max {d.max():+.3f}')
    print("\nSMOKE OK. Ponha SMOKE = False e rode de novo para o numero real.")
    sys.exit()

# ── 4. run completo ──────────────────────────────────────────────────────────
OUT = f"/kaggle/working/visual_feats_train_{LANG}.npz"
store, skipped, t0 = {}, 0, time.time()
for i, row in enumerate(rows):
    try:
        img = Image.open(os.path.join(IMAGE_DIR, row["image_name"])).convert("RGB")
    except Exception:
        skipped += 1; continue
    off, f = score_response(img, row["prompt"], row["response"])
    if f is None:
        skipped += 1; continue
    store[row["id"] + "|off"] = off
    store[row["id"] + "|f"]   = f
    if (i + 1) % 100 == 0:
        r = (i + 1) / (time.time() - t0)
        print(f"  {i+1}/{len(rows)}  {r:.1f}/s  eta {(len(rows)-i-1)/r/60:.0f} min  "
              f"skipped {skipped}", flush=True)
np.savez_compressed(OUT, **store)
print(f"\n{len(store)//2} exemplos salvos, {skipped} pulados, "
      f"{(time.time()-t0)/60:.1f} min -> {OUT}")
gc.collect(); torch.cuda.empty_cache()

# ── 5. a sonda: e ISTO que responde a pergunta ───────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

d = np.load(OUT)
by_id = {json.loads(l)["id"]: json.loads(l)
         for l in open(f"{DISTRIB}/shroom-vision.train.{LANG}.labeled.jsonl")}
X, y, g = [], [], []
for k in [k for k in d.files if k.endswith("|f")]:
    rid = k[:-2]
    row = by_id[rid]; n = len(row["response"]); gold = np.zeros(n)
    for s in row["labels"]:
        gold[s["start"]:s["end"]] = np.maximum(gold[s["start"]:s["end"]], s["prob"])
    for (a, b), f in zip(d[rid + "|off"], d[k]):
        X.append(f); y.append(bool(b > a and gold[a:b].max() > 0)); g.append(rid)
X, y, g = np.array(X), np.array(y), np.array(g)

NAMES = ["logp_img", "ent_img", "logp_noimg", "ent_noimg", "d_logp", "d_ent"]
def probe(cols):
    out = []
    for seed in range(5):
        tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=seed).split(X, y, g))
        mu, sd = X[tr][:, cols].mean(0), X[tr][:, cols].std(0) + 1e-8
        c = LogisticRegression(max_iter=2000, class_weight="balanced")
        c.fit((X[tr][:, cols] - mu) / sd, y[tr])
        out.append(roc_auc_score(y[te], c.predict_proba((X[te][:, cols] - mu) / sd)[:, 1]))
    return float(np.mean(out)), float(np.std(out))

print("\n" + "=" * 62)
print("  RESPOSTA — output-level likelihood, AUROC por feature")
print("=" * 62)
for i, nm in enumerate(NAMES):
    m, s = probe([i])
    flag = "   <-- a diferenca contrastiva (o paper diz 0.485)" if nm == "d_logp" else ""
    print(f"    {nm:12s} {m:.3f} +- {s:.3f}{flag}")
m, s = probe(list(range(6)))
print(f"    {'todas as 6':12s} {m:.3f} +- {s:.3f}")
print("=" * 62)
print("  Baixe visual_feats_train_en.npz antes de fechar a sessao.")
print("=" * 62) in parse(raws.get(row["id"], ""), row["response"])])
    print("GOLD:", [row["response"][x["start"]:x["end"]] for x in row["labels"]])

SyntaxError: invalid syntax. Perhaps you forgot a comma? (488241193.py, line 137)